# Ablation Comparison

Compare recovery and hallucination rate across ablation variants.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT / 'src'))
sys.path.insert(0, str(REPO_ROOT / 'experiments'))
sys.path.insert(0, str(REPO_ROOT))

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
DATASET         = 'pond'
MODEL           = 'gemma-3-27b'
ABLATIONS       = ['1', '2', '3', '4', '5', '6']  # subset as needed
ABLATION_DATES  = {a: None for a in ABLATIONS}     # None = most recent run
BASELINE_DATE   = None                              # None = most recent baseline

In [ ]:
import pandas as pd
from analysis.loaders import load_extraction, load_ablation, load_combined_judgements, load_ground_truth
from analysis.metrics import recovery_rate, hallucination_rate
from analysis.plots import recovery_bar
from run_extraction import load_dataset_config

config = load_dataset_config(DATASET)
ground_truth_df = load_ground_truth(config)

STRICT_MATCHING = {'entity': ['name', 'location']}
FUZZY_MATCHING  = {'value': ['value']}

## Load all ablations

In [ ]:
rows = []

# Baseline
baseline_records = load_extraction(DATASET, MODEL, BASELINE_DATE)
baseline_df = pd.DataFrame(baseline_records)
stats = recovery_rate(baseline_df, ground_truth_df, strict_matching=STRICT_MATCHING, fuzzy_matching=FUZZY_MATCHING)
rows.append({'name': 'baseline', **stats})

# Ablations
for ablation_n in ABLATIONS:
    try:
        records = load_ablation(DATASET, ablation_n, MODEL, ABLATION_DATES.get(ablation_n))
        df = pd.DataFrame(records)
        stats = recovery_rate(df, ground_truth_df, strict_matching=STRICT_MATCHING, fuzzy_matching=FUZZY_MATCHING)
        rows.append({'name': f'ablation {ablation_n}', **stats})
    except FileNotFoundError:
        print(f'Ablation {ablation_n}: no results found, skipping.')

summary_df = pd.DataFrame(rows).set_index('name')
summary_df.round(3)

## Recovery comparison

In [ ]:
fig = recovery_bar(summary_df[['recall', 'precision']], title=f'{DATASET} — {MODEL}')
Path('figures').mkdir(exist_ok=True)
fig.savefig('figures/ablation_recovery.pdf', bbox_inches='tight')
fig.show()

## Summary table

In [ ]:
summary_df.round(3)